# Train a YOLO Model

In [ ]:
import torch
import torch.nn as nn
from ultralytics.utils.tal import TaskAlignedAssigner, make_anchors
from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.utils.loss import v8DetectionLoss
from ultralytics.utils import LOGGER, RANK
from ultralytics.utils.metrics import bbox_iou
from typing import Any, Dict, Tuple

# ---------------------------------------------------------------------------
# 1. Custom Assigner (with Negative Label Correction)
# ---------------------------------------------------------------------------

class CustomAssigner(nn.Module):
    """
    A custom task-aligned assigner that implements "Negative Label Correction".
    It drops background samples where the model predicts an object with high confidence,
    assuming these are unlabeled positives from a partially-labeled dataset.
    This prevents the model from being penalized for finding new, correct objects.
    """

    def __init__(self, topk: int = 10, num_classes: int = 80, alpha: float = 0.5, beta: float = 6.0, eps: float = 1e-9, 
                 correction_thresh: float = 0.3):
        """
        Initialize the CustomAssigner.
        
        Args:
            topk (int): The number of top candidates to consider for assignment.
            num_classes (int): The number of object classes.
            alpha (float): The alpha parameter for the alignment metric.
            beta (float): The beta parameter for the alignment metric.
            eps (float): A small value to prevent division by zero.
            correction_thresh (float): Confidence threshold. Background samples with a max class score
                                     ABOVE this value will be dropped from the loss calculation.
                                     A value of 0.0 disables this feature.
        """
        super().__init__()
        self.topk = topk
        self.num_classes = num_classes
        self.alpha = alpha
        self.beta = beta
        self.eps = eps
        
        # --- Our custom parameter for Negative Label Correction ---
        self.correction_thresh = correction_thresh
        
        if RANK in (-1, 0):
            print(f"✅ CustomAssigner initialized.")
            if self.correction_thresh > 0.0:
                print(f"   - Negative Label Correction: ENABLED (threshold = {self.correction_thresh})")
            else:
                print(f"   - Negative Label Correction: DISABLED")

    @torch.no_grad()
    def forward(self, pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt):
        """
        Computes the assignment and then applies our Negative Label Correction logic.
        """
        self.bs = pd_scores.shape[0]
        self.n_max_boxes = gt_bboxes.shape[1]
        device = gt_bboxes.device

        if self.n_max_boxes == 0:
            return (
                torch.full_like(pd_scores[..., 0], self.num_classes),
                torch.zeros_like(pd_bboxes),
                torch.zeros_like(pd_scores),
                torch.zeros_like(pd_scores[..., 0]),
                torch.zeros_like(pd_scores[..., 0]),
            )

        try:
            # Perform the standard assignment logic by calling the internal _forward method
            results = self._forward(pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt)
        except torch.cuda.OutOfMemoryError:
            LOGGER.warning("CUDA OutOfMemoryError in CustomAssigner, using CPU")
            cpu_tensors = [t.cpu() for t in (pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt)]
            results = self._forward(*cpu_tensors)
            results = tuple(t.to(device) for t in results)

        target_labels, target_bboxes, target_scores, fg_mask, target_gt_idx = results

        if self.training and self.correction_thresh > 0.0:
            # --- NEGATIVE LABEL CORRECTION LOGIC ---
            
            # 1. Identify all initial background regions
            bg_mask = ~fg_mask
            bg_indices = torch.where(bg_mask)

            if bg_indices[0].numel() > 0:
                # 2. Get the model's multi-class prediction scores for these background regions
                bg_scores_all_classes = pd_scores[bg_indices]
                
                # 3. Find the max score for each background anchor (its "objectness" confidence)
                bg_max_scores, _ = bg_scores_all_classes.max(dim=-1)

                # 4. Identify which samples have a confidence ABOVE our correction threshold.
                # These are the samples we suspect are unlabeled positives.
                mask_to_drop = bg_max_scores > self.correction_thresh
                
                # 5. Get the original indices of the samples to drop.
                batch_idx_to_drop = bg_indices[0][mask_to_drop]
                anchor_idx_to_drop = bg_indices[1][mask_to_drop]

                # 6. Exclude these suspected unlabeled positives from the loss calculation
                # by setting their target score to -1.
                target_scores[batch_idx_to_drop, anchor_idx_to_drop] = -1.0

        return target_labels, target_bboxes, target_scores, fg_mask, target_gt_idx

    # --- ALL HELPER METHODS ARE COPIED DIRECTLY FROM THE ORIGINAL ULTRALYTICS IMPLEMENTATION ---

    def _forward(self, pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt):
        """Internal forward pass for assignment."""
        mask_pos, align_metric, overlaps = self.get_pos_mask(
            pd_scores, pd_bboxes, gt_labels, gt_bboxes, anc_points, mask_gt
        )
        target_gt_idx, fg_mask, mask_pos = self.select_highest_overlaps(mask_pos, overlaps, self.n_max_boxes)
        target_labels, target_bboxes, target_scores = self.get_targets(gt_labels, gt_bboxes, target_gt_idx, fg_mask)
        align_metric *= mask_pos
        pos_align_metrics = align_metric.amax(dim=-1, keepdim=True)
        pos_overlaps = (overlaps * mask_pos).amax(dim=-1, keepdim=True)
        norm_align_metric = (align_metric * pos_overlaps / (pos_align_metrics + self.eps)).amax(-2).unsqueeze(-1)
        target_scores = target_scores * norm_align_metric
        return target_labels, target_bboxes, target_scores, fg_mask.bool(), target_gt_idx

    def get_pos_mask(self, pd_scores, pd_bboxes, gt_labels, gt_bboxes, anc_points, mask_gt):
        """Get positive mask."""
        mask_in_gts = self.select_candidates_in_gts(anc_points, gt_bboxes)
        align_metric, overlaps = self.get_box_metrics(pd_scores, pd_bboxes, gt_labels, gt_bboxes, mask_in_gts * mask_gt)
        mask_topk = self.select_topk_candidates(align_metric, topk_mask=mask_gt.expand(-1, -1, self.topk).bool())
        mask_pos = mask_topk * mask_in_gts * mask_gt
        return mask_pos, align_metric, overlaps

    def get_box_metrics(self, pd_scores, pd_bboxes, gt_labels, gt_bboxes, mask_gt):
        """Compute alignment metric."""
        na = pd_bboxes.shape[-2]
        mask_gt = mask_gt.bool()
        overlaps = torch.zeros([self.bs, self.n_max_boxes, na], dtype=pd_bboxes.dtype, device=pd_bboxes.device)
        bbox_scores = torch.zeros([self.bs, self.n_max_boxes, na], dtype=pd_scores.dtype, device=pd_scores.device)
        ind = torch.zeros([2, self.bs, self.n_max_boxes], dtype=torch.long)
        ind[0] = torch.arange(end=self.bs).view(-1, 1).expand(-1, self.n_max_boxes)
        ind[1] = gt_labels.squeeze(-1)
        bbox_scores[mask_gt] = pd_scores[ind[0], :, ind[1]][mask_gt]
        pd_boxes = pd_bboxes.unsqueeze(1).expand(-1, self.n_max_boxes, -1, -1)[mask_gt]
        gt_boxes = gt_bboxes.unsqueeze(2).expand(-1, -1, na, -1)[mask_gt]
        overlaps[mask_gt] = self.iou_calculation(gt_boxes, pd_boxes)
        align_metric = bbox_scores.pow(self.alpha) * overlaps.pow(self.beta)
        return align_metric, overlaps

    def iou_calculation(self, gt_bboxes, pd_bboxes):
        """Calculate IoU."""
        return bbox_iou(gt_bboxes, pd_bboxes, xywh=False, CIoU=True).squeeze(-1).clamp_(0)

    def select_topk_candidates(self, metrics, topk_mask=None):
        """Select top-k candidates."""
        topk_metrics, topk_idxs = torch.topk(metrics, self.topk, dim=-1, largest=True)
        if topk_mask is None:
            topk_mask = (topk_metrics.max(-1, keepdim=True)[0] > self.eps).expand_as(topk_idxs)
        topk_idxs.masked_fill_(~topk_mask, 0)
        count_tensor = torch.zeros(metrics.shape, dtype=torch.int8, device=topk_idxs.device)
        ones = torch.ones_like(topk_idxs[:, :, :1], dtype=torch.int8, device=topk_idxs.device)
        for k in range(self.topk):
            count_tensor.scatter_add_(-1, topk_idxs[:, :, k : k + 1], ones)
        count_tensor.masked_fill_(count_tensor > 1, 0)
        return count_tensor.to(metrics.dtype)

    def get_targets(self, gt_labels, gt_bboxes, target_gt_idx, fg_mask):
        """Compute target labels, bboxes and scores."""
        batch_ind = torch.arange(end=self.bs, dtype=torch.int64, device=gt_labels.device)[..., None]
        target_gt_idx = target_gt_idx + batch_ind * self.n_max_boxes
        target_labels = gt_labels.long().flatten()[target_gt_idx]
        target_bboxes = gt_bboxes.view(-1, gt_bboxes.shape[-1])[target_gt_idx]
        target_labels.clamp_(0)
        target_scores = torch.zeros((target_labels.shape[0], target_labels.shape[1], self.num_classes),
                                    dtype=torch.int64, device=target_labels.device)
        target_scores.scatter_(2, target_labels.unsqueeze(-1), 1)
        fg_scores_mask = fg_mask[:, :, None].repeat(1, 1, self.num_classes)
        target_scores = torch.where(fg_scores_mask > 0, target_scores, 0)
        return target_labels, target_bboxes, target_scores

    @staticmethod
    def select_candidates_in_gts(xy_centers, gt_bboxes, eps=1e-9):
        """Select anchors in ground truth boxes."""
        n_anchors = xy_centers.shape[0]
        bs, n_boxes, _ = gt_bboxes.shape
        lt, rb = gt_bboxes.view(-1, 1, 4).chunk(2, 2)
        bbox_deltas = torch.cat((xy_centers[None] - lt, rb - xy_centers[None]), dim=2).view(bs, n_boxes, n_anchors, -1)
        return bbox_deltas.amin(3).gt_(eps)

    @staticmethod
    def select_highest_overlaps(mask_pos, overlaps, n_max_boxes):
        """Handle multiple gt assignments."""
        fg_mask = mask_pos.sum(-2)
        if fg_mask.max() > 1:
            mask_multi_gts = (fg_mask.unsqueeze(1) > 1).expand(-1, n_max_boxes, -1)
            max_overlaps_idx = overlaps.argmax(1)
            is_max_overlaps = torch.zeros(mask_pos.shape, dtype=mask_pos.dtype, device=mask_pos.device)
            is_max_overlaps.scatter_(1, max_overlaps_idx.unsqueeze(1), 1)
            mask_pos = torch.where(mask_multi_gts, is_max_overlaps, mask_pos).float()
            fg_mask = mask_pos.sum(-2)
        target_gt_idx = mask_pos.argmax(-2)
        return target_gt_idx, fg_mask, mask_pos


# ---------------------------------------------------------------------------
# 2. Custom Loss Class (A clean way to package the change)
# ---------------------------------------------------------------------------

class CustomV8DetectionLoss(v8DetectionLoss):
    """
    A custom version of v8DetectionLoss that uses our CustomAssigner
    to implement Negative Label Correction for partially-labeled datasets.
    """
    def __init__(self, model):
        super().__init__(model)
        if not hasattr(self, 'bbox_decode'):
            self.bbox_decode = self.decode_bboxes
        
        # Get the custom argument from the training configuration (e.g., from trainer.train(correction_thresh=0.3))
        correction_thresh = getattr(model.args, 'correction_thresh', 0.3)
        
        # Replace the default assigner with our custom one
        self.assigner = CustomAssigner(topk=10, num_classes=self.nc, alpha=0.5, beta=6.0, 
                                       correction_thresh=correction_thresh)
        if RANK in (-1, 0):
            print("✅ CustomV8DetectionLoss initialized and assigner replaced.")

    def __call__(self, preds, batch):
        """
        Calculates the loss using the custom assigner.
        The -1 target scores from the assigner are automatically handled by the cls_mask.
        """
        loss = torch.zeros(3, device=self.device)  # box, cls, dfl
        feats = preds[1] if isinstance(preds, tuple) else preds
        pred_distri, pred_scores = torch.cat([xi.view(feats[0].shape[0], self.no, -1) for xi in feats], 2).split(
            (self.reg_max * 4, self.nc), 1)

        pred_scores = pred_scores.permute(0, 2, 1).contiguous()
        pred_distri = pred_distri.permute(0, 2, 1).contiguous()

        dtype = pred_scores.dtype
        batch_size = pred_scores.shape[0]
        imgsz = torch.tensor(feats[0].shape[2:], device=self.device, dtype=dtype) * self.stride[0]
        anchor_points, stride_tensor = make_anchors(feats, self.stride, 0.5)

        targets = torch.cat((batch['batch_idx'].view(-1, 1), batch['cls'].view(-1, 1), batch['bboxes']), 1)
        targets = self.preprocess(targets.to(self.device), batch_size, scale_tensor=imgsz[[1, 0, 1, 0]])
        gt_labels, gt_bboxes = targets.split((1, 4), 2)
        mask_gt = gt_bboxes.sum(2, keepdim=True).gt_(0)

        pred_bboxes = self.bbox_decode(anchor_points, pred_distri)

        _, target_bboxes, target_scores, fg_mask, _ = self.assigner(
            pred_scores.detach().sigmoid(),
            (pred_bboxes.detach() * stride_tensor).type(gt_bboxes.dtype),
            anchor_points * stride_tensor,
            gt_labels,
            gt_bboxes,
            mask_gt)
        
        target_scores_sum = max(target_scores[target_scores > 0].sum(), 1)

        # Classification loss: Create a mask to select only targets that are NOT -1.0
        # This correctly handles positives, true negatives, and ignores corrected negatives.
        cls_mask = (target_scores != -1.0)
        
        loss_cls_unnormalized = self.bce(pred_scores[cls_mask], target_scores[cls_mask].to(dtype))

        num_pos = fg_mask.sum()
        if num_pos > 0:
            loss[1] = loss_cls_unnormalized.sum() / num_pos
        
        # Bbox loss (this part is correct and unaffected)
        if fg_mask.sum():
            target_bboxes /= stride_tensor
            try:
                loss[0], loss[2] = self.bbox_loss(
                    pred_distri, pred_bboxes, anchor_points, target_bboxes, target_scores, target_scores_sum, fg_mask)
            except TypeError:
                loss[0], loss[2] = self.bbox_loss(
                    pred_distri, pred_bboxes, anchor_points, target_bboxes, target_scores, target_scores_sum)

        loss[0] *= self.hyp.box
        loss[1] *= self.hyp.cls
        loss[2] *= self.hyp.dfl

        return loss.sum() * batch_size, loss.detach()


# ---------------------------------------------------------------------------
# 3. Custom Trainer (Injects the custom loss function)
# ---------------------------------------------------------------------------

class CustomTrainer(DetectionTrainer):
    """
    This trainer injects our custom loss function into the training process.
    """

    def _setup_train(self, world_size):
        """
        Overrides the training setup to replace the model's loss function.
        """
        # First, run the standard setup from the parent class.
        super()._setup_train(world_size)

        # Now, replace the standard `v8DetectionLoss` with our custom version.
        if RANK in (-1, 0):
            print("✅ Replacing the model's criterion with CustomV8DetectionLoss.")
        self.model.criterion = CustomV8DetectionLoss(self.model)

        # Ensure the assigner is on the correct device.
        self.model.criterion.assigner.to(self.device)

In [ ]:
import torch
import torch.nn as nn
import math
from typing import Any, Dict, Tuple

# Ultralytics imports
from ultralytics.engine.trainer import BaseTrainer
from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.utils.loss import v8DetectionLoss
from ultralytics.utils.tal import make_anchors
from ultralytics.utils.metrics import bbox_iou
from ultralytics.utils import LOGGER, RANK

# ---------------------------------------------------------------------------
# 1. Custom Assigner (Stable - No Changes Needed)
# ---------------------------------------------------------------------------
class CustomAssigner(nn.Module):
    """
    Custom task-aligned assigner for YOLO models.
    Optionally implements Negative Label Correction by dropping background samples
    with high model confidence, assuming they are unlabeled positives.
    """
    def __init__(self, topk: int = 10, num_classes: int = 80, alpha: float = 0.5, beta: float = 6.0, eps: float = 1e-9, 
                 correction_thresh: float = 0.0):
        """
        Initialize the CustomAssigner.
        Args:
            topk (int): Number of top candidates for assignment.
            num_classes (int): Number of object classes.
            alpha (float): Alpha parameter for alignment metric.
            beta (float): Beta parameter for alignment metric.
            eps (float): Small value to prevent division by zero.
            correction_thresh (float): Confidence threshold for negative label correction.
        """
        super().__init__()
        self.topk = topk
        self.num_classes = num_classes
        self.alpha = alpha
        self.beta = beta
        self.eps = eps
        self.correction_thresh = correction_thresh

    @torch.no_grad()
    def forward(self, pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt):
        """
        Forward pass for assignment and optional negative label correction.
        Args:
            pd_scores: Predicted class scores.
            pd_bboxes: Predicted bounding boxes.
            anc_points: Anchor points.
            gt_labels: Ground truth labels.
            gt_bboxes: Ground truth bounding boxes.
            mask_gt: Mask for valid ground truth.
        Returns:
            target_labels, target_bboxes, target_scores, fg_mask, target_gt_idx
        """
        self.bs = pd_scores.shape[0]
        self.n_max_boxes = gt_bboxes.shape[1]
        device = gt_bboxes.device
        if self.n_max_boxes == 0:
            # No ground truth boxes, return default tensors
            return (torch.full_like(pd_scores[..., 0], self.num_classes),
                    torch.zeros_like(pd_bboxes),
                    torch.zeros_like(pd_scores),
                    torch.zeros_like(pd_scores[..., 0]),
                    torch.zeros_like(pd_scores[..., 0]))
        try:
            # Standard assignment logic
            results = self._forward(pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt)
        except torch.cuda.OutOfMemoryError:
            LOGGER.warning("CUDA OutOfMemoryError in CustomAssigner, retrying on CPU.")
            results = self._forward(*[t.cpu() for t in (pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt)])
            results = tuple(t.to(device) for t in results)
        target_labels, target_bboxes, target_scores, fg_mask, target_gt_idx = results
        if self.training and self.correction_thresh > 0.0:
            # Negative Label Correction: drop background anchors with high confidence
            bg_mask = ~fg_mask
            bg_indices = torch.where(bg_mask)
            if bg_indices[0].numel() > 0:
                bg_scores_all_classes = pd_scores[bg_indices]
                bg_max_scores, _ = bg_scores_all_classes.max(dim=-1)
                mask_to_drop = bg_max_scores > self.correction_thresh
                batch_idx_to_drop = bg_indices[0][mask_to_drop]
                anchor_idx_to_drop = bg_indices[1][mask_to_drop]
                target_scores[batch_idx_to_drop, anchor_idx_to_drop] = -1.0
        return target_labels, target_bboxes, target_scores, fg_mask, target_gt_idx

    def _forward(self, pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt):
        """
        Internal forward pass for assignment logic.
        """
        # Get positive mask, alignment metric, and overlaps for assignment
        mask_pos, align_metric, overlaps = self.get_pos_mask(
            pd_scores, pd_bboxes, gt_labels, gt_bboxes, anc_points, mask_gt
        )
        # For anchors assigned to multiple GTs, select the one with highest overlap
        target_gt_idx, fg_mask, mask_pos = self.select_highest_overlaps(
            mask_pos, overlaps, self.n_max_boxes
        )
        # Compute target labels, bounding boxes, and scores for loss calculation
        target_labels, target_bboxes, target_scores = self.get_targets(
            gt_labels, gt_bboxes, target_gt_idx, fg_mask
        )
        # Refine alignment metric for positive samples only
        align_metric *= mask_pos
        pos_align_metrics = align_metric.amax(dim=-1, keepdim=True)
        pos_overlaps = (overlaps * mask_pos).amax(dim=-1, keepdim=True)
        # Normalize alignment metric for each anchor
        norm_align_metric = (
            align_metric * pos_overlaps / (pos_align_metrics + self.eps)
        ).amax(-2).unsqueeze(-1)
        # Scale target scores by normalized alignment metric
        target_scores = target_scores * norm_align_metric
        # Return targets and masks for loss computation
        return target_labels, target_bboxes, target_scores, fg_mask.bool(), target_gt_idx

    def get_pos_mask(self, pd_scores, pd_bboxes, gt_labels, gt_bboxes, anc_points, mask_gt):
        """
        Get positive mask for assignment.
        """
        # Select anchor points that are inside ground truth boxes
        mask_in_gts = self.select_candidates_in_gts(anc_points, gt_bboxes)
        # Compute alignment metric and overlaps for assignment, only for valid GTs
        align_metric, overlaps = self.get_box_metrics(
            pd_scores, pd_bboxes, gt_labels, gt_bboxes, mask_in_gts * mask_gt
        )
        # Select top-k candidates for each GT using the alignment metric
        mask_topk = self.select_topk_candidates(
            align_metric, topk_mask=mask_gt.expand(-1, -1, self.topk).bool()
        )
        # Final positive mask: top-k, inside GT, and valid GT mask
        mask_pos = mask_topk * mask_in_gts * mask_gt
        return mask_pos, align_metric, overlaps

    def get_box_metrics(self, pd_scores, pd_bboxes, gt_labels, gt_bboxes, mask_gt):
        """
        Compute alignment metric and overlaps for assignment.
        """
        # Number of anchors
        na = pd_bboxes.shape[-2]
        # Ensure mask_gt is boolean for indexing
        mask_gt = mask_gt.bool()
        # Initialize overlaps and bbox_scores tensors
        overlaps = torch.zeros([self.bs, self.n_max_boxes, na], dtype=pd_bboxes.dtype, device=pd_bboxes.device)
        bbox_scores = torch.zeros([self.bs, self.n_max_boxes, na], dtype=pd_scores.dtype, device=pd_scores.device)
        # Prepare indices for batch and class
        ind = torch.zeros([2, self.bs, self.n_max_boxes], dtype=torch.long)
        ind[0] = torch.arange(end=self.bs).view(-1, 1).expand(-1, self.n_max_boxes)
        ind[1] = gt_labels.squeeze(-1)
        # Assign predicted class scores to bbox_scores for valid GTs
        bbox_scores[mask_gt] = pd_scores[ind[0], :, ind[1]][mask_gt]
        # Expand predicted and GT boxes for IoU calculation
        pd_boxes = pd_bboxes.unsqueeze(1).expand(-1, self.n_max_boxes, -1, -1)[mask_gt]
        gt_boxes = gt_bboxes.unsqueeze(2).expand(-1, -1, na, -1)[mask_gt]
        # Calculate IoU overlaps for valid GTs
        overlaps[mask_gt] = self.iou_calculation(gt_boxes, pd_boxes)
        # Compute alignment metric using bbox_scores and overlaps
        align_metric = bbox_scores.pow(self.alpha) * overlaps.pow(self.beta)
        return align_metric, overlaps

    def iou_calculation(self, gt_bboxes, pd_bboxes):
        """
        Calculate IoU between ground truth and predicted boxes.
        """
        return bbox_iou(gt_bboxes, pd_bboxes, xywh=False, CIoU=True).squeeze(-1).clamp_(0)

    def select_topk_candidates(self, metrics, topk_mask=None):
        """
        Select top-k candidates for assignment based on the alignment metric.
        """
        # Get top-k indices and values along the anchor dimension
        topk_metrics, topk_idxs = torch.topk(metrics, self.topk, dim=-1, largest=True)
        if topk_mask is None:
            # If no mask provided, create a mask where top-k metric is above eps
            topk_mask = (topk_metrics.max(-1, keepdim=True)[0] > self.eps).expand_as(topk_idxs)
        # Mask out invalid indices
        topk_idxs.masked_fill_(~topk_mask, 0)
        # Create a tensor to count selected anchors
        count_tensor = torch.zeros(metrics.shape, dtype=torch.int8, device=topk_idxs.device)
        ones = torch.ones_like(topk_idxs[:, :, :1], dtype=torch.int8, device=topk_idxs.device)
        # Scatter add to count how many times each anchor is selected
        for k in range(self.topk):
            count_tensor.scatter_add_(-1, topk_idxs[:, :, k : k + 1], ones)
        # Remove anchors selected more than once (should not happen, but for safety)
        count_tensor.masked_fill_(count_tensor > 1, 0)
        # Return as float mask for further processing
        return count_tensor.to(metrics.dtype)

    def get_targets(self, gt_labels, gt_bboxes, target_gt_idx, fg_mask):
        """
        Compute target labels, bounding boxes, and scores for loss calculation.
        """
        # Compute flattened indices for batch and GT
        batch_ind = torch.arange(end=self.bs, dtype=torch.int64, device=gt_labels.device)[..., None]
        target_gt_idx = target_gt_idx + batch_ind * self.n_max_boxes
        # Gather assigned labels and boxes
        target_labels = gt_labels.long().flatten()[target_gt_idx]
        target_bboxes = gt_bboxes.view(-1, gt_bboxes.shape[-1])[target_gt_idx]
        target_labels.clamp_(0)
        # Create one-hot encoded class scores
        target_scores = torch.zeros((target_labels.shape[0], target_labels.shape[1], self.num_classes), dtype=torch.int64, device=target_labels.device)
        target_scores.scatter_(2, target_labels.unsqueeze(-1), 1)
        # Mask out background anchors
        fg_scores_mask = fg_mask[:, :, None].repeat(1, 1, self.num_classes)
        target_scores = torch.where(fg_scores_mask > 0, target_scores, 0)
        return target_labels, target_bboxes, target_scores

    @staticmethod
    def select_candidates_in_gts(xy_centers, gt_bboxes, eps=1e-9):
        """
        Select anchor points that are inside ground truth boxes.
        """
        n_anchors = xy_centers.shape[0]
        bs, n_boxes, _ = gt_bboxes.shape
        # Split GT boxes into left-top and right-bottom
        lt, rb = gt_bboxes.view(-1, 1, 4).chunk(2, 2)
        # Calculate deltas from anchor to box corners
        bbox_deltas = torch.cat((xy_centers[None] - lt, rb - xy_centers[None]), dim=2).view(bs, n_boxes, n_anchors, -1)
        # Anchor is inside GT if all deltas > eps
        return bbox_deltas.amin(3).gt_(eps)

    @staticmethod
    def select_highest_overlaps(mask_pos, overlaps, n_max_boxes):
        """
        For anchors assigned to multiple GTs, select the one with highest overlap.
        """
        fg_mask = mask_pos.sum(-2)
        if fg_mask.max() > 1:
            # If an anchor is assigned to multiple GTs, keep only the one with highest IoU
            mask_multi_gts = (fg_mask.unsqueeze(1) > 1).expand(-1, n_max_boxes, -1)
            max_overlaps_idx = overlaps.argmax(1)
            is_max_overlaps = torch.zeros(mask_pos.shape, dtype=mask_pos.dtype, device=mask_pos.device)
            is_max_overlaps.scatter_(1, max_overlaps_idx.unsqueeze(1), 1)
            mask_pos = torch.where(mask_multi_gts, is_max_overlaps, mask_pos).float()
            fg_mask = mask_pos.sum(-2)
        # For each anchor, get the GT index with the highest overlap
        target_gt_idx = mask_pos.argmax(-2)
        return target_gt_idx, fg_mask, mask_pos

# ---------------------------------------------------------------------------
# 2. Custom Loss Class (CORRECTED)
# ---------------------------------------------------------------------------
class CustomV8DetectionLoss(v8DetectionLoss):
    """
    Custom loss class for YOLO v8 that uses CustomAssigner for assignment.
    This enables negative label correction and other custom assignment logic.
    """
    def __init__(self, model):
        """
        Initialize the custom loss with a standard constructor.
        Replaces the default assigner with CustomAssigner.
        """
        super().__init__(model)
        if not hasattr(self, 'bbox_decode'):
            self.bbox_decode = self.decode_bboxes
        # Initialize with a default assigner. The trainer will configure it.
        self.assigner = CustomAssigner(topk=10, num_classes=self.nc, alpha=0.5, beta=6.0)

    def __call__(self, preds, batch):
        """
        Compute the YOLO loss using the custom assigner.
        Args:
            preds: Model predictions.
            batch: Batch dictionary with targets.
        Returns:
            Total loss and detached loss tensor.
        """
        # Initialize loss tensor for [box, cls, dfl] components
        loss = torch.zeros(3, device=self.device)
        # Get model features (feats) from predictions
        feats = preds[1] if isinstance(preds, tuple) else preds
        # Concatenate and split predictions into distribution and class scores
        pred_distri, pred_scores = torch.cat([xi.view(feats[0].shape[0], self.no, -1) for xi in feats], 2).split((self.reg_max * 4, self.nc), 1)
        pred_scores = pred_scores.permute(0, 2, 1).contiguous()
        pred_distri = pred_distri.permute(0, 2, 1).contiguous()
        dtype = pred_scores.dtype
        batch_size = pred_scores.shape[0]
        # Compute image size tensor for scaling
        imgsz = torch.tensor(feats[0].shape[2:], device=self.device, dtype=dtype) * self.stride[0]
        # Generate anchor points and stride tensor
        anchor_points, stride_tensor = make_anchors(feats, self.stride, 0.5)
        # Prepare targets: concatenate batch indices, class labels, and bounding boxes
        targets = torch.cat((batch['batch_idx'].view(-1, 1), batch['cls'].view(-1, 1), batch['bboxes']), 1)
        # Preprocess targets for assignment
        targets = self.preprocess(targets.to(self.device), batch_size, scale_tensor=imgsz[[1, 0, 1, 0]])
        gt_labels, gt_bboxes = targets.split((1, 4), 2)
        # Mask for valid ground truth boxes
        mask_gt = gt_bboxes.sum(2, keepdim=True).gt_(0)
        # Decode predicted bounding boxes
        pred_bboxes = self.bbox_decode(anchor_points, pred_distri)
        # Assign targets using the custom assigner (with negative label correction)
        _, target_bboxes, target_scores, fg_mask, _ = self.assigner(
            pred_scores.detach().sigmoid(),
            (pred_bboxes.detach() * stride_tensor).type(gt_bboxes.dtype),
            anchor_points * stride_tensor,
            gt_labels,
            gt_bboxes,
            mask_gt)
        # Compute sum of positive target scores (avoid division by zero)
        target_scores_sum = max(target_scores[target_scores > 0].sum(), 1)
        # Mask for valid classification targets (not dropped by correction)
        cls_mask = (target_scores != -1.0)
        # Compute unnormalized classification loss (binary cross-entropy)
        loss_cls_unnormalized = self.bce(pred_scores[cls_mask], target_scores[cls_mask].to(dtype))
        num_pos = fg_mask.sum()
        # Normalize classification loss by number of positive samples
        if num_pos > 0:
            loss[1] = loss_cls_unnormalized.sum() / num_pos
        # If there are foreground (positive) samples, compute box and DFL loss
        if fg_mask.sum():
            target_bboxes /= stride_tensor
            try:
            loss[0], loss[2] = self.bbox_loss(pred_distri, pred_bboxes, anchor_points, target_bboxes, target_scores, target_scores_sum, fg_mask)
            except TypeError:
            loss[0], loss[2] = self.bbox_loss(pred_distri, pred_bboxes, anchor_points, target_bboxes, target_scores, target_scores_sum)
        # Scale losses by hyperparameters
        loss[0] *= self.hyp.box
        loss[1] *= self.hyp.cls
        loss[2] *= self.hyp.dfl
        # Return total loss (summed) and detached loss tensor
        return loss.sum() * batch_size, loss.detach()

# ---------------------------------------------------------------------------
# 3. Custom Trainer (CORRECTED)
# ---------------------------------------------------------------------------
class CustomTrainer(DetectionTrainer):
    """
    Custom trainer for YOLO that supports custom arguments and adaptive threshold scheduling.
    Handles custom arguments for negative label correction and mAP-based scheduling.
    """
    def __init__(self, overrides=None, _callbacks=None):
        """
        Initializes the trainer, intercepting and storing custom arguments.
        """
        # Define the custom keys our trainer will use.
        self.custom_keys = [
            'correction_thresh_start',
            'correction_thresh_end',
            'map_start_target',
            'map_end_target'
        ]
        # Pop our custom args from the overrides dict and store them on the instance.
        if overrides:
            for key in self.custom_keys:
                if key in overrides:
                    setattr(self, key, overrides.pop(key))
        # Initialize the parent class with the "clean" overrides dict.
        super().__init__(overrides=overrides, _callbacks=_callbacks)

    def _setup_train(self, world_size):
        """
        Overrides the training setup to replace the loss function and configure it.
        Replaces the default criterion with CustomV8DetectionLoss and configures the assigner.
        Registers a callback for adaptive threshold scheduling based on mAP.
        """
        super()._setup_train(world_size)
        if RANK in (-1, 0):
            LOGGER.info("✅ Replacing the model's criterion with CustomV8DetectionLoss.")
        # Replace the default criterion.
        self.model.criterion = CustomV8DetectionLoss(self.model)
        # Configure the assigner's starting threshold using the custom argument.
        start_thresh = getattr(self, 'correction_thresh_start', 0.0)
        self.model.criterion.assigner.correction_thresh = start_thresh
        self.model.criterion.assigner.to(self.device)
        # Register the callback to adapt the threshold based on mAP.
        self.add_callback("on_validation_end", self._map_adaptive_scheduler)
        if RANK in (-1, 0):
            LOGGER.info("✅ Registered callback for mAP-driven adaptive threshold.")
            LOGGER.info(f"   Initial `correction_thresh` set to {start_thresh:.4f}")

    def _map_adaptive_scheduler(self, trainer):
        """
        Callback to adjust `correction_thresh` based on the latest validation mAP@50.
        Schedules the threshold between start and end values as mAP improves.
        """
        # Get scheduler parameters from the instance attributes we saved.
        start_thresh = getattr(self, 'correction_thresh_start', 0.0)
        end_thresh = getattr(self, 'correction_thresh_end', start_thresh)
        if start_thresh == end_thresh:
            if self.epoch == 1 and RANK in (-1, 0):
                 if start_thresh > 0.0:
                     LOGGER.info(f"💡 Negative Correction: Using FIXED threshold of {start_thresh:.4f}")
                 else:
                     LOGGER.info(f"💡 Negative Correction: DISABLED")
            return
        map_start_target = getattr(self, 'map_start_target', 0.1)
        map_end_target = getattr(self, 'map_end_target', 0.8)
        current_map = trainer.validator.metrics.box.map
        if current_map is None or current_map == 0.0:
            new_thresh = start_thresh
        else:
            progress = (current_map - map_start_target) / (map_end_target - map_start_target)
            progress = max(0.0, min(1.0, progress))
            new_thresh = start_thresh - (start_thresh - end_thresh) * progress
        self.model.criterion.assigner.correction_thresh = new_thresh
        if RANK in (-1, 0):
            LOGGER.info(f"💡 mAP Adaptive Threshold: Val mAP50 is {current_map:.4f}. `correction_thresh` set to {new_thresh:.4f} for next epoch.")

In [2]:
data_dir = "E:/JordanP/Click-a-Coral/data/reduced/MDBC_Transects"
dataset = f"{data_dir}/YOLO_Detection_Dataset/data.yaml"
project = f"{data_dir}/results"

### Deconstructing the New mAP50-95 Schedule

Let's trace the story of this new, more rigorous training run, which ties the adaptive threshold to the strict `mAP50-95` metric.

---

#### **Phase 1: The Effective Start (mAP50-95 is 0.0 → 0.05)**

*   **Scenario:** The model is producing very rough boxes or is just beginning to converge. Its `mAP50-95` is below `0.05`.
*   **System's Logic:** "The model cannot yet localize objects precisely. I will stick to the proven starting strategy."
*   **Result:** `correction_thresh` is held **constant at 0.35**.

---

#### **Phase 2: The Precision-Driven Transition (mAP50-95 is 0.05 → 0.15)**

*   **Scenario:** The model's `mAP50-95` has crossed the `0.05` milestone. This is a significant achievement, proving it can generate boxes with decent IoU across multiple thresholds.
*   **System's Logic:** "The model is demonstrating high-quality localization. I will begin to trust its predictions more aggressively as this metric improves."
*   **Result:** `correction_thresh` will smoothly decrease from `0.35` down towards `0.1` as the `mAP50-95` score climbs from `0.05` to `0.15`. This is the core adaptive phase, driven by a much more meaningful signal of model quality.

---

#### **Phase 3: The Expert Endgame (mAP50-95 is > 0.15)**

*   **Scenario:** The model's `mAP50-95` has exceeded the `0.15` "expert" milestone, which is a very strong result for a difficult, partially labeled dataset.
*   **System's Logic:** "The model is a precision expert. Enable maximum trust to find the hardest unlabeled examples."
*   **Result:** `correction_thresh` is held **constant at 0.1** for the rest of training.

---

By switching to `mAP50-95`, you are making the adaptive system "smarter" and more demanding. It now requires proof of high-quality predictions before it starts aggressively modifying the training targets, which should lead to a more stable and ultimately more accurate final model.

In [ ]:
from ultralytics import YOLO

args = dict(
    model="yolov8n.pt",
    data=dataset,
    project=project,
    name="NM_Loss_Adaptive_Multiclass_25",
    task='detect',
    epochs=100,
    patience=10,
    half=True,
    imgsz=640,
    single_cls=False,
    plots=True,
    batch=32,
    workers=0,
    correction_thresh_start=0.3,
    correction_thresh_end=0.1,
    map_start_target=0.05,
    map_end_target=0.15,
)

trainer = CustomTrainer(overrides=args)
trainer.train()

Ultralytics 8.3.101  Python-3.10.16 torch-2.6.0+cu118 CUDA:0 (Tesla T4, 16384MiB)
engine\trainer: task=detect, mode=train, model=yolov8n.pt, data=E:/JordanP/Click-a-Coral/data/reduced/MDBC_Transects/YOLO_Detection_Dataset/data.yaml, epochs=100, time=None, patience=10, batch=32, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=0, project=E:/JordanP/Click-a-Coral/data/reduced/MDBC_Transects/results, name=NM_Loss_Adaptive_Multiclass_25, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=True, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False

train: Scanning E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\train\labels.cache... 4615 images, 0 backgrounds, 0 corrupt: 100%|██████████| 4615/4615 [00:00<?, ?it/s]

train: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\train\images\6761625_122544.jpg: 1 duplicate labels removed
train: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\train\images\6761625_122549.jpg: 1 duplicate labels removed
train: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\train\images\6761625_122552.jpg: 1 duplicate labels removed
train: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\train\images\6761625_122566.jpg: 1 duplicate labels removed
train: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\train\images\6761625_122568.jpg: 1 duplicate labels removed
train: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\train\images\6761625_122572.jpg: 1 duplicate labels removed
train: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Data


val: Scanning E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\valid\labels.cache... 578 images, 0 backgrounds, 0 corrupt: 100%|██████████| 578/578 [00:00<?, ?it/s]

val: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\valid\images\6761625_108524.jpg: 1 duplicate labels removed
val: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\valid\images\6761625_122540.jpg: 1 duplicate labels removed
val: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\valid\images\6761625_122743.jpg: 1 duplicate labels removed
val: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\valid\images\6761625_122765.jpg: 1 duplicate labels removed
val: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\valid\images\6761625_122809.jpg: 1 duplicate labels removed
val: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\valid\images\6761625_122817.jpg: 1 duplicate labels removed
val: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\valid\imag

...

                   all        578        823      0.318      0.485       0.23      0.112

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     25/100      4.35G      1.784      0.603      1.497         11        640: 100%|██████████| 145/145 [05:57<00:00,  2.47s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 10/10 [00:36<00:00,  3.60s/it]

                   all        578        823      0.456      0.405      0.258      0.132



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     26/100      4.35G       1.78     0.6096      1.506         81        640:  64%|██████▍   | 93/145 [04:33<02:10,  2.51s/it]